# CENG 467 - Natural Language Understanding and Generation
## Take-Home Midterm - Complete Notebook
**Student:** Ceren Nur Arıkoğlu

**Student Id:** 330201101

**Instructor:** Prof. Dr. Aytuğ Onan

# Global setup

In [ ]:
!pip install transformers datasets evaluate accelerate rouge_score seqeval nltk scikit-learn -q

import torch
import numpy as np
import pandas as pd
import random
import nltk
from datasets import load_dataset

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"System Ready. Device: {device}")

System Ready. Device: cuda


#Question 1 – Representation Learning in Text Classification

In [ ]:
# sst2 dataset load
dataset_q1 = load_dataset("glue", "sst2")
train_df = pd.DataFrame(dataset_q1['train'])
val_df = pd.DataFrame(dataset_q1['validation'])

print(f"Dataset Loaded: {len(train_df)} train, {len(val_df)} validation samples.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Dataset Loaded: 67349 train, 872 validation samples.


**3 Different preprocessing strategies compared**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def run_experiment(name, vectorizer):
    X_train = vectorizer.fit_transform(train_df['sentence'])
    X_val = vectorizer.transform(val_df['sentence'])
    model = LogisticRegression(max_iter=1000).fit(X_train, train_df['label'])
    preds = model.predict(X_val)
    return {
        "Experiment": name,
        "Accuracy": accuracy_score(val_df['label'], preds),
        "Macro-F1": f1_score(val_df['label'], preds, average='macro')
    }

# Experiment list (Normalization, Stopword, Truncation effects)
results_prep = []
results_prep.append(run_experiment("Basic (No Norm)", TfidfVectorizer(lowercase=False)))
results_prep.append(run_experiment("With Normalization", TfidfVectorizer(lowercase=True)))
results_prep.append(run_experiment("Without Stopwords", TfidfVectorizer(stop_words='english')))
results_prep.append(run_experiment("With Truncation (500 features)", TfidfVectorizer(max_features=500)))

print(pd.DataFrame(results_prep))

                       Experiment  Accuracy  Macro-F1
0                 Basic (No Norm)  0.824541  0.823939
1              With Normalization  0.824541  0.823939
2               Without Stopwords  0.803899  0.802106
3  With Truncation (500 features)  0.741972  0.741087


**Model comparison a classical model based on TF-IDF (Logistic Regression), a neural model such as BiLSTM, and a transformer-based model (DistilBERT)**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from collections import Counter
import numpy as np
import re

#since normalization gave us the best preprocess result, we'll use that preprocess strategy
def preprocess_text(text):
    # 1.(Normalization)
    text = text.lower()
    # 2. Cleaning up punctuation marks
    text = re.sub(r"[^\w\s]", "", text)
    return text

# All models operate on the same preprocessed text (clean column).
train_df["clean"] = train_df["sentence"].apply(preprocess_text)
val_df["clean"] = val_df["sentence"].apply(preprocess_text)

# from now on we'll use clean sentences
train_texts = train_df['clean'].tolist()
val_texts = val_df['clean'].tolist()

def get_performance(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, average='macro')
    }

#------ MODEL 1: SPARSE -----
#BASED ON TF-IDF (LOGISTIC REGRESSION) RESULTS

tfidf = TfidfVectorizer(max_features=5000)
X_train_s = tfidf.fit_transform(train_texts)
X_val_s = tfidf.transform(val_texts)
model_sparse = LogisticRegression(max_iter=1000).fit(X_train_s, train_df['label'])
sparse_preds = model_sparse.predict(X_val_s)
sparse_results = get_performance(val_df['label'], sparse_preds)


# --- FOR BI-LSTM SPECIAL VOCAB AND DATA LOADER ---

all_tokens = " ".join(train_texts).split()
vocab_counts = Counter(all_tokens)
# We take the 10k most frequently occurring words (consistent with the max_features value of the Sparse model).
vocab = {word: i+2 for i, (word, _) in enumerate(vocab_counts.most_common(5000))}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def word_tokenize(texts, vocab, max_len=64):
    tokenized = []
    for text in texts:
        ids = [vocab.get(w, 1) for w in text.split()][:max_len]
        ids += [0] * (max_len - len(ids))
        tokenized.append(ids)
    return torch.tensor(tokenized)

X_train_d = word_tokenize(train_texts, vocab)
X_val_d = word_tokenize(val_texts, vocab)
y_train_d = torch.tensor(train_df['label'].tolist())
y_val_d = torch.tensor(val_df['label'].tolist())

train_loader = DataLoader(TensorDataset(X_train_d, y_train_d), batch_size=32, shuffle=True)

# --- DENSE MODEL ARCHITECTURE (BiLSTM) ---
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # Randomly initialized word-level embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Bidirectional LSTM layer
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        # Classification Layer (Hidden_dim * 2 because bidirectional)
        self.fc = nn.Linear(hidden_dim * 2, 2)

    def forward(self, x):
        embedded = self.embedding(x)
        # LSTM output: output, (hidden, cell)
        _, (hidden, _) = self.lstm(embedded)
        # (forward ve backward) concatenate
        cat_hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(cat_hidden)

# --- MODEL 2: DENSE (BiLSTM)  ---

#  device (GPU/CPU)
bilstm_model = BiLSTMClassifier(len(vocab), 128, 64).to(device)
optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(2): # for Fair comparison 2 epoch
    bilstm_model.train()
    for b_x, b_y in train_loader:
        b_x, b_y = b_x.to(device), b_y.to(device)
        optimizer.zero_grad()
        loss = criterion(bilstm_model(b_x), b_y)
        loss.backward()
        optimizer.step()


bilstm_model.eval()
with torch.no_grad():
    bilstm_preds = torch.argmax(bilstm_model(X_val_d.to(device)), dim=1).cpu().numpy()

dense_results = get_performance(val_df['label'], bilstm_preds)



# --- MODEL 3: CONTEXTUAL (DistilBERT) ---
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# We ensure split consistency by recreating the dataset from pandas df.
from datasets import Dataset
train_ds = Dataset.from_pandas(train_df[['clean', 'label']].rename(columns={'clean': 'sentence'}))
val_ds = Dataset.from_pandas(val_df[['clean', 'label']].rename(columns={'clean': 'sentence'}))

def tokenize_fn(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_ds.map(tokenize_fn, batched=True)
tokenized_val = val_ds.map(tokenize_fn, batched=True)

model_bert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

t_args = TrainingArguments(
    output_dir="results",
    num_train_epochs=2, # equal with BiLSTM
    per_device_train_batch_size=32,
    report_to="none",
    eval_strategy="no"
)

trainer = Trainer(model=model_bert, args=t_args, train_dataset=tokenized_train)
trainer.train()

bert_preds = np.argmax(trainer.predict(tokenized_val).predictions, axis=1)
context_results = get_performance(val_df['label'], bert_preds)

# --- COMPARISON SUMMARY ---
comparison_df = pd.DataFrame({
    "Model (Representation)": ["Sparse (TF-IDF)", "Dense (BiLSTM)", "Contextual (BERT)"],
    "Accuracy": [sparse_results["Accuracy"], dense_results["Accuracy"], context_results["Accuracy"]],
    "Macro-F1": [sparse_results["Macro-F1"], dense_results["Macro-F1"], context_results["Macro-F1"]]
})
print("\n--- FINAL FAIR COMPARISON ---")
print(comparison_df)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- FINAL FAIR COMPARISON ---
  Model (Representation)  Accuracy  Macro-F1
0        Sparse (TF-IDF)  0.815367  0.814997
1         Dense (BiLSTM)  0.810780  0.810624
2      Contextual (BERT)  0.899083  0.898929


In [ ]:
# 1. Filtering out errored predictions.
val_df['bert_pred'] = bert_preds
# take errored ones
errors = val_df[val_df['label'] != val_df['bert_pred']].head(5)

print("\n--- ERROR ANALYSIS (Top 5 BERT Errors) ---")
for i, row in errors.iterrows():
    print(f"sentence: {row['sentence']}")
    print(f"real label: {row['label']} | model prediction: {row['bert_pred']}")
    print("-" * 30)


--- ERROR ANALYSIS (Top 5 BERT Errors) ---
sentence: holden caulfield did it better . 
real label: 0 | model prediction: 1
------------------------------
sentence: the script kicks in , and mr. hartley 's distended pace and foot-dragging rhythms follow . 
real label: 0 | model prediction: 1
------------------------------
sentence: this one is definitely one to skip , even for horror movie fanatics . 
real label: 0 | model prediction: 1
------------------------------
sentence: though it 's become almost redundant to say so , major kudos go to leigh for actually casting people who look working-class . 
real label: 1 | model prediction: 0
------------------------------
sentence: you wo n't like roger , but you will quickly recognize him . 
real label: 0 | model prediction: 1
------------------------------


# Question 2 - Named Entity Recognition

In [ ]:
!pip install datasets transformers seqeval evaluate pytorch-crf

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification
import evaluate
import numpy as np
from tqdm import tqdm
from torchcrf import CRF

dataset_q2 = load_dataset("tomaarsen/conll2003")
label_list = dataset_q2["train"].features["ner_tags"].feature.names
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# CONVERT DATA TO BERT FORMAT
tokenized_datasets = dataset_q2.map(tokenize_and_align_labels, batched=True)
tokenized_train = tokenized_datasets["train"]
tokenized_val = tokenized_datasets["validation"]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

**Classical/Hybrid (BiLSTM-CRF)**

In [ ]:
class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tagset_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 64, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(128, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)

    def forward(self, x, labels=None, mask=None):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        emissions = self.fc(x)

        if labels is not None:
            # must provide the mask parameter to the CRF.
            return -self.crf(emissions, labels, mask=mask)
        else:
            # Masks should also be used when decoding.
            return self.crf.decode(emissions, mask=mask)

In [ ]:
from collections import Counter
# A. (Vocabulary) preparation
# Since BiLSTM is trained from scratch, we need to create a word index.
all_tokens = [t for tokens in dataset_q2["train"]["tokens"] for t in tokens]
counter = Counter(all_tokens)
vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common())}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def text_to_indices(tokens):
    return [vocab.get(t, vocab["<UNK>"]) for t in tokens]

# B. Data Collation (Padding & Masking)
def collate_fn(batch):
    # convert sentences to indeces the tensor
    inputs = [torch.tensor(text_to_indices(item["tokens"])) for item in batch]
    labels = [torch.tensor(item["ner_tags"]) for item in batch]

    # Padding step according to longest sentence
    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=0)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=0)

    # create mask: real words 1, paddings 0 (critical for CRF)
    mask = (inputs_padded != 0)

    return inputs_padded.to(device), labels_padded.to(device), mask.to(device)

# Dataloaders
train_loader = DataLoader(dataset_q2["train"], batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(dataset_q2["validation"], batch_size=16, collate_fn=collate_fn)

# C. Model Setup
model_hybrid = BiLSTM_CRF(len(vocab), len(label_list)).to(device)
optimizer = optim.Adam(model_hybrid.parameters(), lr=0.001)

# D. Training Cycle (2 Epoch - consistent with BERT)
for epoch in range(2):
    model_hybrid.train()
    total_loss = 0
    for inputs, labels, mask in tqdm(train_loader):
        optimizer.zero_grad()
        loss = model_hybrid(inputs, labels, mask=mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

100%|██████████| 878/878 [00:53<00:00, 16.26it/s]


Epoch 1 Loss: 113.5963


100%|██████████| 878/878 [00:37<00:00, 23.36it/s]

Epoch 2 Loss: 40.9737


**Transformer Based Model (BERT)**

In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)
    true_labels = [[label_list[l] for l in label if l != -100] for label in labels]
    true_preds = [[label_list[p] for p, l in zip(prediction, label) if l != -100]
                  for prediction, label in zip(predictions, labels)]
    return seqeval.compute(predictions=true_preds, references=true_labels)

In [ ]:
model_bert = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(label_list))
training_args = TrainingArguments(
    output_dir="ner_results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.115024,0.040685,"{'precision': 0.9557474365893146, 'recall': 0.9640718562874252, 'f1': 0.9598915989159892, 'number': 1837}","{'precision': 0.895505617977528, 'recall': 0.8644251626898047, 'f1': 0.8796909492273731, 'number': 922}","{'precision': 0.8792982456140351, 'recall': 0.9343773303504848, 'f1': 0.9060014461315979, 'number': 1341}","{'precision': 0.9611650485436893, 'recall': 0.9674267100977199, 'f1': 0.9642857142857142, 'number': 1842}",0.930422,0.942949,0.936643,0.989720
2,0.020790,0.037035,"{'precision': 0.9709589041095891, 'recall': 0.9646162221012521, 'f1': 0.967777170944839, 'number': 1837}","{'precision': 0.8827877507919747, 'recall': 0.9067245119305857, 'f1': 0.8945960406634563, 'number': 922}","{'precision': 0.9205052005943536, 'recall': 0.9239373601789709, 'f1': 0.9222180870859693, 'number': 1341}","{'precision': 0.9699248120300752, 'recall': 0.9804560260586319, 'f1': 0.9751619870410367, 'number': 1842}",0.945318,0.951363,0.948331,0.991180


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1756, training_loss=0.052402283987857755, metrics={'train_runtime': 384.9828, 'train_samples_per_second': 72.944, 'train_steps_per_second': 4.561, 'total_flos': 701093358405576.0, 'train_loss': 0.052402283987857755, 'epoch': 2.0})

In [ ]:
# BiLSTM-CRF results
model_hybrid.eval()
all_preds_hybrid = []
all_true_hybrid = []
with torch.no_grad():
    for inputs, labels, mask in val_loader:
        paths = model_hybrid(inputs, mask=mask)
        for i in range(len(paths)):
            true_len = mask[i].sum().item()
            true_tags = labels[i][:true_len].cpu().numpy()
            all_true_hybrid.append([label_list[t] for t in true_tags])
            all_preds_hybrid.append([label_list[p] for p in paths[i]])

In [ ]:
crf_results = seqeval.compute(
    predictions=all_preds_hybrid,
    references=all_true_hybrid
)

crf_precision = crf_results["overall_precision"]
crf_recall = crf_results["overall_recall"]
crf_f1 = crf_results["overall_f1"]

bert_results = trainer.evaluate()

bert_precision = bert_results["eval_overall_precision"]
bert_recall = bert_results["eval_overall_recall"]
bert_f1 = bert_results["eval_overall_f1"]

import pandas as pd

comparison_df = pd.DataFrame({
    "Model": ["BiLSTM-CRF", "BERT"],
    "Precision": [crf_precision, bert_precision],
    "Recall": [crf_recall, bert_recall],
    "F1-score": [crf_f1, bert_f1]
})

print("\nFINAL PERFORMANCE COMPARISON")
print(comparison_df)


FINAL PERFORMANCE COMPARISON
        Model  Precision    Recall  F1-score
0  BiLSTM-CRF   0.848312  0.651296  0.736862
1        BERT   0.945318  0.951363  0.948331


# Question 3 - Text Summarization

In [ ]:
!pip install sumy
!pip install datasets transformers
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import numpy as np
import pandas as pd
import nltk
from datasets import load_dataset
from transformers import pipeline
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from tqdm import tqdm

dataset_q3 = load_dataset("cnn_dailymail", "3.0.0",split="test[:20]")
articles = dataset_q3["article"]
references = dataset_q3["highlights"]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


**Extractive Method (TextRank)**

In [ ]:
def get_extractive_summaries(text_list):
    summaries = []
    summarizer = TextRankSummarizer()
    for text in tqdm(text_list, desc="Extractive"):
        parser = PlaintextParser.from_string(text, Tokenizer("english"))
        # select 3 most important sentences
        summary_sentences = summarizer(parser.document, 3)
        summaries.append(" ".join([str(s) for s in summary_sentences]))
    return summaries

extractive_summaries = get_extractive_summaries(articles)

**Abstractive Method (BART)**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
def get_abstractive_summary_manual(text):
    # Convert the text to a format the model can understand. (with the limit of 1024 token)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)

    # generate the summary (for BART optimized parameters)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=130,
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    # convert numerical datas back to text
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

abstractive_summaries = []

for art in tqdm(articles, desc="Abstractive Processing"):
    # empty text control
    if not art.strip():
        abstractive_summaries.append("")
        continue

    summary = get_abstractive_summary_manual(art)
    abstractive_summaries.append(summary)

Abstractive Processing: 100%|██████████| 20/20 [01:41<00:00,  5.06s/it]


**EVALUATION OF METRICES**

In [ ]:
!pip install rouge_score evaluate bert_score
import evaluate

# upload metrices
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

def evaluate_summaries(predictions, references):
    # ROUGE
    r = rouge.compute(predictions=predictions, references=references)

    # BLEU
    # - BLEU requires a list of reference lists for each prediction.
    b = bleu.compute(predictions=predictions, references=[[r] for r in references])

    # METEOR
    m = meteor.compute(predictions=predictions, references=references)

    # BERTSCORE
    bs = bertscore.compute(
        predictions=predictions,
        references=references,
        lang="en",
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

    return {
        "R1": r["rouge1"],
        "R2": r["rouge2"],
        "RL": r["rougeL"],
        "BLEU": b["bleu"],
        "METEOR": m["meteor"],
        "BERTScore": np.mean(bs["f1"])
    }

# evaluate results
ext_metrics = evaluate_summaries(extractive_summaries, references)
abs_metrics = evaluate_summaries(abstractive_summaries, references)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import pandas as pd

# update comparison_data dictionary according to the new keys that function returns (R1, R2, RL...)
comparison_data = {
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "METEOR", "BERTScore"],
    "Extractive (TextRank)": [
        ext_metrics["R1"],
        ext_metrics["R2"],
        ext_metrics["RL"],
        ext_metrics["BLEU"],
        ext_metrics["METEOR"],
        ext_metrics["BERTScore"]
    ],
    "Abstractive (BART)": [
        abs_metrics["R1"],
        abs_metrics["R2"],
        abs_metrics["RL"],
        abs_metrics["BLEU"],
        abs_metrics["METEOR"],
        abs_metrics["BERTScore"]
    ]
}

df_metrics = pd.DataFrame(comparison_data)
print("\n--- SUMMARIZATION PERFORMANCE COMPARISON ---")
print(df_metrics)


--- SUMMARIZATION PERFORMANCE COMPARISON ---
      Metric  Extractive (TextRank)  Abstractive (BART)
0    ROUGE-1               0.224074            0.359901
1    ROUGE-2               0.063861            0.163684
2    ROUGE-L               0.155286            0.281989
3       BLEU               0.033939            0.130115
4     METEOR               0.288793            0.346544
5  BERTScore               0.852576            0.880121


In [ ]:
# Let's print out the first three examples in detail and compare them.
print("\n" + "="*80)
print("             QUALITATIVE ANALYSIS: EXTRACTIVE VS ABSTRACTIVE")
print("="*80)

for i in range(3):
    print(f"\n--- EXAMPLE {i+1} ---")

    # source text (just first 300 characters)
    print(f"\n[SOURCE ARTICLE]:\n{articles[i][:400]}...")

    # gold standard (reference)
    print(f"\n[GOLD STANDARD REFERENCE]:\n{references[i]}")

    # Extractive output
    print(f"\n[EXTRACTIVE - TextRank]:\n{extractive_summaries[i]}")

    # Abstractive output
    print(f"\n[ABSTRACTIVE - BART]:\n{abstractive_summaries[i]}")

    print("\n" + "-"*50)



             QUALITATIVE ANALYSIS: EXTRACTIVE VS ABSTRACTIVE

--- EXAMPLE 1 ---

[SOURCE ARTICLE]:
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also acce...

[GOLD STANDARD REFERENCE]:
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

[EXTRACTIVE - TextRank]:
"As Palestine formally becomes a State Party to the Rome Statute today, the world is also a step closer to ending a long era of impunity and injustice," he said, according to an ICC news release. Judge Kuniko

# Question 4 - Machine Translation

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

dataset_q4 = load_dataset("bentrevett/multi30k", split="test[:30]")
source_texts = dataset_q4["en"]
target_references = dataset_q4["de"]


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

**Transformer Baseed Model (SOTA)**

In [ ]:
device = 0 if torch.cuda.is_available() else -1
transformer_model_name = "Helsinki-NLP/opus-mt-en-de"
tokenizer_trans = AutoTokenizer.from_pretrained(transformer_model_name)
model_trans = AutoModelForSeq2SeqLM.from_pretrained(transformer_model_name).to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

**Seq2Seq + Attention (MarianMT's old versions)**

In [ ]:
leg_name = "google-t5/t5-small"
tokenizer_leg = AutoTokenizer.from_pretrained(leg_name)
model_leg = AutoModelForSeq2SeqLM.from_pretrained(leg_name).to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def translate_manual(text, model, tokenizer, is_t5=False):
    # add prefix for T5
    if is_t5:
        text = "translate English to German: " + text

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    # Seq2Seq Generation (where Attention mechanism comes into play )
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=100,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
transformer_preds = [translate_manual(t, model_trans, tokenizer_trans) for t in tqdm(source_texts)]
legacy_preds = [translate_manual(t, model_leg, tokenizer_leg, is_t5=True) for t in tqdm(source_texts)]

100%|██████████| 30/30 [00:07<00:00,  4.13it/s]


In [ ]:
!pip install evaluate rouge_score sacrebleu bert_score
import nltk
import evaluate

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
chrf = evaluate.load("chrf")
bertscore = evaluate.load("bertscore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


**Evaluating Metrices**

In [ ]:
def evaluate_translation_suite(preds, refs):
    # BLEU
    b = bleu.compute(predictions=preds, references=[[r] for r in refs])

    # METEOR
    m = meteor.compute(predictions=preds, references=refs)

    # ChrF: character level n-gram (ideal for german suffixes)
    c = chrf.compute(predictions=preds, references=[[r] for r in refs])

    # BERTScore: semantic similarity
    bs = bertscore.compute(predictions=preds, references=refs, lang="de", device=torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    return {
        "BLEU": b["bleu"],
        "METEOR": m["meteor"],
        "ChrF": c["score"],
        "BERTScore": np.mean(bs["f1"])
    }

# evaluate metrices
trans_metrics = evaluate_translation_suite(transformer_preds, target_references)
leg_metrics = evaluate_translation_suite(legacy_preds, target_references)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
df_mt_comparison = pd.DataFrame({
    "Metric": ["BLEU", "METEOR", "ChrF", "BERTScore"],
    "Seq2Seq (T5/Legacy)": [leg_metrics["BLEU"], leg_metrics["METEOR"], leg_metrics["ChrF"], leg_metrics["BERTScore"]],
    "Transformer (OPUS)": [trans_metrics["BLEU"], trans_metrics["METEOR"], trans_metrics["ChrF"], trans_metrics["BERTScore"]]
})

print("\n--- MACHINE TRANSLATION PERFORMANCE ---")
print(df_mt_comparison)


--- MACHINE TRANSLATION PERFORMANCE ---
      Metric  Seq2Seq (T5/Legacy)  Transformer (OPUS)
0       BLEU             0.290911            0.430028
1     METEOR             0.604018            0.732661
2       ChrF            56.423300           66.116915
3  BERTScore             0.880920            0.906823


In [ ]:
print("\n--- QUALITATIVE TRANSLATION EXAMPLE ---")
idx = 0
print(f"Source (EN): {source_texts[idx]}")
print(f"Reference (DE): {target_references[idx]}")
print(f"Seq2Seq Output: {legacy_preds[idx]}")
print(f"Transformer Output: {transformer_preds[idx]}")



--- QUALITATIVE TRANSLATION EXAMPLE ---
Source (EN): A man in an orange hat starring at something.
Reference (DE): Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
Seq2Seq Output: Ein Mann in einem orangen Hut, der auf etwas dreht.
Transformer Output: Ein Mann in einem orangenen Hut, mit etwas.


# Question 5 - Language Modeling

In [ ]:
from datasets import load_dataset
import torch
import math
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from collections import Counter, defaultdict
from tqdm import tqdm
#we are using wikitest dataset
dataset_q5_train = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
dataset_q5_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
#empty rows are being cleared and concatanate sentences
q5_train_text = " ".join([line for line in dataset_q5_train["text"] if len(line.strip()) > 0])
q5_test_text = " ".join([line for line in dataset_q5_test["text"] if len(line.strip()) > 0])
train_tokens = q5_train_text.split()
test_tokens = q5_test_text.split()

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

**N-gram model(Bigram)**

In [ ]:
def train_bigram(tokens):
    model = defaultdict(Counter)
    for i in range(len(tokens)-1):
        w1, w2 = tokens[i], tokens[i+1]
        model[w1][w2] += 1

    probs = defaultdict(dict)
    for w1, counts in model.items():
        total = sum(counts.values())
        for w2, count in counts.items():
            probs[w1][w2] = count / total
    return probs

bigram_model = train_bigram(train_tokens)

**Transformer Based Model (GPT-2)**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"
tokenizer_gpt2 = GPT2Tokenizer.from_pretrained(model_name)
model_gpt2 = GPT2LMHeadModel.from_pretrained(model_name).to(device)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
def generate_bigram(model, start_word, length=15):
    current_word = start_word
    sentence = [current_word]
    for _ in range(length):
        if current_word not in model: break
        next_words = list(model[current_word].keys())
        next_probs = list(model[current_word].values())
        current_word = np.random.choice(next_words, p=next_probs)
        sentence.append(current_word)
    return " ".join(sentence)

def generate_gpt2(prompt, model, tokenizer):
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    # We are blocking the warning by adding an attention mask.
    attention_mask = torch.ones(inputs.shape, device=device)
    outputs = model.generate(
        inputs,
        attention_mask=attention_mask,
        max_length=40,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=2,
        do_sample=True, # For more natural production
        top_k=50
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

**Perplexity Analysis**

In [ ]:
def calculate_gpt2_perplexity(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        outputs = model(inputs["input_ids"], labels=inputs["input_ids"])
    return math.exp(outputs.loss.item())

def calculate_bigram_perplexity(model, test_tokens):
    log_sum = 0
    N = len(test_tokens) - 1
    # For unfamiliar words, we make the smoothing a bit more realistic.
    unknown_prob = 1 / len(train_tokens)

    for i in range(N):
        w1, w2 = test_tokens[i], test_tokens[i+1]
        prob = model.get(w1, {}).get(w2, unknown_prob)
        log_sum += math.log2(prob)
    return 2 ** (-log_sum / N)

In [ ]:
sample_test_tokens = test_tokens[100:600] # 500 words from test set
sample_test_text = " ".join(sample_test_tokens)

bigram_ppl = calculate_bigram_perplexity(bigram_model, sample_test_tokens)
gpt2_ppl = calculate_gpt2_perplexity(sample_test_text, model_gpt2, tokenizer_gpt2)

print(f"\n--- REVISED LANGUAGE MODELING RESULTS ---")
print(f"Bigram Perplexity: {bigram_ppl:.2f}")
print(f"GPT-2 Perplexity: {gpt2_ppl:.2f}")

print(f"\nBigram Sample: {generate_bigram(bigram_model, 'The')}")
print(f"GPT-2 Sample: {generate_gpt2('The scientific discovery of', model_gpt2, tokenizer_gpt2)}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



--- REVISED LANGUAGE MODELING RESULTS ---
Bigram Perplexity: 2138.93
GPT-2 Perplexity: 23.69

Bigram Sample: The Wild Thornberrys ( 121 km ) , Corbet within weeks . Infantry within the city
GPT-2 Sample: The scientific discovery of these processes cannot be justified as a defense of a natural law, or any natural rule, in favour of the natural sciences (which will always be supported for themselves by empirical work in
